In [1]:
import glob
print(len(glob.glob('/kaggle/input/datasets/aratisutar/phasea-run2/P_L*')), 'folders')

50 folders


In [2]:
import glob
for p in sorted(glob.glob('/kaggle/input/datasets/aratisutar/*')):
    n = len(glob.glob(f'{p}/P_L*'))
    if n:
        print(n, p.split('/')[-1])
        for lvl in range(6):
            c = len(glob.glob(f'{p}/P_L{lvl}_*'))
            if c: print(f'    L{lvl}: {c}')

30 pha-run1
    L0: 10
    L1: 10
    L2: 10
50 phasea-run2
    L0: 10
    L1: 10
    L2: 10
    L3: 10
    L4: 10
60 phasea-run3
    L0: 10
    L1: 10
    L2: 10
    L3: 10
    L4: 10
    L5: 10


In [3]:
import numpy as np, glob
from PIL import Image
rng = np.random.default_rng(0)

def load(p): return np.array(Image.open(p).convert('RGB')).astype(float)

BASE = '/kaggle/input/datasets/aratisutar/phasea-run3'
K = 6
NAMES = ['FFHQ faces', 'CelebA portraits', 'Animal faces',
         'Painted portraits', 'Single objects', 'Landscapes']

def load_level(lvl, n=10, k=K):
    S, T = [], []
    for i in range(n):
        fs = sorted(glob.glob(f'{BASE}/P_L{lvl}_{i:05d}/*.png'))[:k]
        if len(fs) < k: continue
        S.append(np.stack([load(f) for f in fs]))
        T.append(load(f'{BASE}/truth_L{lvl}_{i:05d}.png'))
    return np.stack(S), np.stack(T)

LVL = {lvl: load_level(lvl) for lvl in range(6)}
for lvl in range(6):
    print(f'L{lvl}', LVL[lvl][0].shape)

L0 (10, 6, 256, 256, 3)
L1 (10, 6, 256, 256, 3)
L2 (10, 6, 256, 256, 3)
L3 (10, 6, 256, 256, 3)
L4 (10, 6, 256, 256, 3)
L5 (10, 6, 256, 256, 3)


In [4]:
# --- the target: reconstruction error (needs ground truth) ---
def error_of(S, T):
    return np.array([np.mean([np.abs(s-t).mean() for s in S1])
                     for S1, t in zip(S, T)])

# --- signals computable WITHOUT ground truth ---
def signals(S1):
    """S1 is (6, 256, 256, 3) - six samples of one image"""
    k = len(S1)
    mu = S1.mean(axis=0)
    sd = S1.std(axis=0).mean(axis=2)        # per-pixel disagreement, greyscale

    # 1. how much do samples disagree, on average
    spread = np.mean([np.abs(S1[a]-S1[b]).mean()
                      for a in range(k) for b in range(a+1,k)])

    # 2. how concentrated is the disagreement
    flat = sd.ravel()
    conc = np.quantile(flat, 0.9) / (flat.mean() + 1e-8)

    # 3. does disagreement sit on edges of the reconstruction?
    g = np.abs(np.gradient(mu.mean(axis=2))[0]) + \
        np.abs(np.gradient(mu.mean(axis=2))[1])
    hi = g >= np.quantile(g, 0.9)
    edge_ratio = sd[hi].mean() / (sd[~hi].mean() + 1e-8)

    # 4. frequency profile of the disagreement
    F = np.abs(np.fft.fft2(sd))**2
    N = sd.shape[0]
    fy, fx = np.meshgrid(np.fft.fftfreq(N)*N, np.fft.fftfreq(N)*N, indexing='ij')
    r = np.sqrt(fy**2 + fx**2)
    low  = F[r < 16].mean()
    high = F[r >= 64].mean()
    hf_frac = high / (low + high + 1e-8)

    return np.array([spread, conc, edge_ratio, hf_frac])

NAMES_SIG = ['spread', 'concentration', 'edge_ratio', 'hf_fraction']

X, Y, LBL = [], [], []
for lvl in range(6):
    S, T = LVL[lvl]
    X.append(np.stack([signals(s) for s in S]))
    Y.append(error_of(S, T))
    LBL += [lvl]*len(S)
X = np.vstack(X); Y = np.concatenate(Y); LBL = np.array(LBL)
print('X', X.shape, ' Y', Y.shape)

print(f"\n{'signal':>15s} {'r with error':>13s}")
for j, nm in enumerate(NAMES_SIG):
    print(f'{nm:>15s} {np.corrcoef(X[:,j], Y)[0,1]:+13.3f}')

X (60, 4)  Y (60,)

         signal  r with error
         spread        +0.619
  concentration        -0.349
     edge_ratio        -0.386
    hf_fraction        -0.454


In [5]:
from itertools import combinations

def r2(cols):
    A = np.column_stack([X[:, c] for c in cols] + [np.ones(len(X))])
    coef, *_ = np.linalg.lstsq(A, Y, rcond=None)
    pred = A @ coef
    return 1 - ((Y-pred)**2).sum() / ((Y-Y.mean())**2).sum()

print(f"{'signals':<40s} {'R2':>7s}")
print(f"{'spread alone':<40s} {r2([0]):7.3f}")
for j in [1,2,3]:
    print(f"{'spread + ' + NAMES_SIG[j]:<40s} {r2([0,j]):7.3f}")
print(f"{'all four':<40s} {r2([0,1,2,3]):7.3f}")
print(f"{'pattern only (no spread)':<40s} {r2([1,2,3]):7.3f}")

signals                                       R2
spread alone                               0.383
spread + concentration                     0.402
spread + edge_ratio                        0.393
spread + hf_fraction                       0.423
all four                                   0.435
pattern only (no spread)                   0.209


In [6]:
print(f"{'level':<20s} {'n':>3s} {'r: spread':>10s} {'r: hf_frac':>11s}")
for lvl in range(6):
    m = LBL == lvl
    print(f'{NAMES[lvl]:<20s} {m.sum():3d} '
          f'{np.corrcoef(X[m,0], Y[m])[0,1]:10.3f} '
          f'{np.corrcoef(X[m,3], Y[m])[0,1]:11.3f}')

level                  n  r: spread  r: hf_frac
FFHQ faces            10      0.946       0.011
CelebA portraits      10      0.896      -0.512
Animal faces          10      0.854       0.079
Painted portraits     10      0.818       0.539
Single objects        10      0.431      -0.039
Landscapes            10      0.614      -0.355


Rank statistics****


In [7]:
K = 6

def rank_hist(S1, T1):
    """where does the truth sit among the samples, per pixel"""
    below = (S1 < T1[None]).sum(axis=0)      # 0..6
    h = np.bincount(below.ravel(), minlength=K+1).astype(float)
    return h / h.sum()

print(f"{'level':<20s} " + ' '.join(f'r{i}' for i in range(K+1)) + '    U-stat')
RANK = {}
for lvl in range(6):
    S, T = LVL[lvl]
    h = np.mean([rank_hist(s, t) for s, t in zip(S, T)], axis=0)
    RANK[lvl] = h
    ends = h[0] + h[K]
    mid  = h[1:K].sum()
    u = ends / (2/(K+1)) - 1          # 0 if flat, >0 if too narrow
    print(f'{NAMES[lvl]:<20s} ' + ' '.join(f'{x:.3f}' for x in h) + f'   {u:+.2f}')

level                r0 r1 r2 r3 r4 r5 r6    U-stat
FFHQ faces           0.184 0.141 0.126 0.115 0.116 0.127 0.191   +0.31
CelebA portraits     0.214 0.140 0.117 0.117 0.119 0.129 0.164   +0.32
Animal faces         0.240 0.143 0.113 0.101 0.099 0.111 0.193   +0.52
Painted portraits    0.213 0.090 0.075 0.076 0.086 0.122 0.338   +0.93
Single objects       0.259 0.124 0.132 0.123 0.112 0.106 0.145   +0.41
Landscapes           0.309 0.094 0.074 0.068 0.079 0.111 0.265   +1.01


In [8]:
def loo_r2(cols):
    """leave-one-out: fit on 59, predict the held-out one"""
    pred = np.zeros(len(Y))
    for i in range(len(Y)):
        m = np.ones(len(Y), bool); m[i] = False
        A = np.column_stack([X[m][:, c] for c in cols] + [np.ones(m.sum())])
        coef, *_ = np.linalg.lstsq(A, Y[m], rcond=None)
        a = np.array([X[i, c] for c in cols] + [1.0])
        pred[i] = a @ coef
    return 1 - ((Y-pred)**2).sum() / ((Y-Y.mean())**2).sum(), pred

def in_sample_r2(cols):
    A = np.column_stack([X[:, c] for c in cols] + [np.ones(len(X))])
    coef, *_ = np.linalg.lstsq(A, Y, rcond=None)
    return 1 - ((Y - A@coef)**2).sum() / ((Y-Y.mean())**2).sum()

print(f"{'model':<34s} {'in-sample':>10s} {'held-out':>10s} {'drop':>8s}")
for lbl, cols in [('spread alone', [0]),
                  ('spread + concentration', [0,1]),
                  ('spread + edge_ratio', [0,2]),
                  ('spread + hf_fraction', [0,3]),
                  ('all four', [0,1,2,3]),
                  ('pattern only, no spread', [1,2,3])]:
    ins = in_sample_r2(cols)
    oos, _ = loo_r2(cols)
    print(f'{lbl:<34s} {ins:10.3f} {oos:10.3f} {ins-oos:+8.3f}')

model                               in-sample   held-out     drop
spread alone                            0.383      0.348   +0.036
spread + concentration                  0.402      0.354   +0.048
spread + edge_ratio                     0.393      0.341   +0.052
spread + hf_fraction                    0.423      0.358   +0.065
all four                                0.435      0.352   +0.082
pattern only, no spread                 0.209      0.109   +0.100


In [9]:
thr = np.quantile(Y, 0.75)
fail = (Y > thr).astype(int)
print(f'threshold: error > {thr:.2f}')
print(f'failures: {fail.sum()} of {len(fail)}')
print('\nby level:')
for lvl in range(6):
    m = LBL == lvl
    print(f'  {NAMES[lvl]:<20s} {fail[m].sum()}/10')

def auc(score, label):
    """probability a random failure scores higher than a random non-failure"""
    pos = score[label == 1]; neg = score[label == 0]
    wins = sum((pos[:, None] > neg[None, :]).sum() for _ in [0])
    ties = sum((pos[:, None] == neg[None, :]).sum() for _ in [0])
    return (wins + 0.5*ties) / (len(pos)*len(neg))

print(f"\n{'signal':>15s} {'AUC':>7s}")
for j, nm in enumerate(NAMES_SIG):
    a = auc(X[:, j], fail)
    print(f'{nm:>15s} {max(a, 1-a):7.3f}')

threshold: error > 16.21
failures: 15 of 60

by level:
  FFHQ faces           1/10
  CelebA portraits     0/10
  Animal faces         5/10
  Painted portraits    6/10
  Single objects       1/10
  Landscapes           2/10

         signal     AUC
         spread   0.926
  concentration   0.708
     edge_ratio   0.778
    hf_fraction   0.738


In [10]:
# per-image ratio = spread / error. Low means overconfident.
RATIO = np.zeros(len(Y))
idx = 0
for lvl in range(6):
    S, T = LVL[lvl]
    for S1, T1 in zip(S, T):
        k = len(S1)
        sp = np.mean([np.abs(S1[a]-S1[b]).mean() for a in range(k) for b in range(a+1,k)])
        er = np.mean([np.abs(s-T1).mean() for s in S1])
        RATIO[idx] = sp/er
        idx += 1

thr_c = np.quantile(RATIO, 0.25)        # worst quarter
miscal = (RATIO < thr_c).astype(int)
print(f'threshold: ratio < {thr_c:.3f}')
print(f'miscalibrated: {miscal.sum()} of {len(miscal)}\n')
print('by level:')
for lvl in range(6):
    m = LBL == lvl
    print(f'  {NAMES[lvl]:<20s} {miscal[m].sum()}/10   (mean ratio {RATIO[m].mean():.3f})')

print(f"\n{'signal':>15s} {'AUC: error':>12s} {'AUC: miscal':>13s}")
for j, nm in enumerate(NAMES_SIG):
    a_err = auc(X[:, j], fail)
    a_mis = auc(X[:, j], miscal)
    print(f'{nm:>15s} {max(a_err,1-a_err):12.3f} {max(a_mis,1-a_mis):13.3f}')

print(f'\noverlap: {(fail & miscal).sum()} images are both high-error and miscalibrated')
print(f'correlation between error and ratio: {np.corrcoef(Y, RATIO)[0,1]:+.3f}')

threshold: ratio < 0.668
miscalibrated: 15 of 60

by level:
  FFHQ faces           0/10   (mean ratio 0.881)
  CelebA portraits     0/10   (mean ratio 0.884)
  Animal faces         1/10   (mean ratio 0.774)
  Painted portraits    7/10   (mean ratio 0.624)
  Single objects       1/10   (mean ratio 0.747)
  Landscapes           6/10   (mean ratio 0.607)

         signal   AUC: error   AUC: miscal
         spread        0.926         0.505
  concentration        0.708         0.664
     edge_ratio        0.778         0.639
    hf_fraction        0.738         0.692

overlap: 8 images are both high-error and miscalibrated
correlation between error and ratio: -0.671


In [11]:
B = 5000
obs_sp = max(auc(X[:,0], miscal), 1-auc(X[:,0], miscal))
obs_hf = max(auc(X[:,3], miscal), 1-auc(X[:,3], miscal))
null = np.zeros(B)
for b in range(B):
    perm = rng.permutation(miscal)
    a = auc(X[:,3], perm)
    null[b] = max(a, 1-a)
p = (null >= obs_hf).mean()
print(f'hf_fraction AUC {obs_hf:.3f}, permutation p = {p:.4f}')
print(f'spread      AUC {obs_sp:.3f}')
print(f'null AUC: median {np.median(null):.3f}, 95th pct {np.quantile(null,0.95):.3f}')

hf_fraction AUC 0.692, permutation p = 0.0268
spread      AUC 0.505
null AUC: median 0.559, 95th pct 0.671


In [12]:
import json
out = {
  'error_corr': {NAMES_SIG[j]: float(np.corrcoef(X[:,j], Y)[0,1]) for j in range(4)},
  'loo_r2': {
    'spread': 0.348, 'spread+conc': 0.354, 'spread+edge': 0.341,
    'spread+hf': 0.358, 'all_four': 0.352, 'pattern_only': 0.109},
  'auc_error': {NAMES_SIG[j]: float(max(auc(X[:,j], fail), 1-auc(X[:,j], fail))) for j in range(4)},
  'auc_miscal': {NAMES_SIG[j]: float(max(auc(X[:,j], miscal), 1-auc(X[:,j], miscal))) for j in range(4)},
  'thresholds': {'error': float(thr), 'ratio': float(thr_c)},
  'within_level_spread_corr': {NAMES[l]: float(np.corrcoef(X[LBL==l,0], Y[LBL==l])[0,1]) for l in range(6)},
  'fail_by_level': {NAMES[l]: int(fail[LBL==l].sum()) for l in range(6)},
  'miscal_by_level': {NAMES[l]: int(miscal[LBL==l].sum()) for l in range(6)},
  'overlap': int((fail & miscal).sum()),
  'corr_error_ratio': float(np.corrcoef(Y, RATIO)[0,1]),
}
with open('/kaggle/working/gtfree_results.json','w') as f:
    json.dump(out, f, indent=2)
print(json.dumps(out, indent=2))

{
  "error_corr": {
    "spread": 0.6190106401897685,
    "concentration": -0.3489787991917791,
    "edge_ratio": -0.3856418134236136,
    "hf_fraction": -0.45353684733670546
  },
  "loo_r2": {
    "spread": 0.348,
    "spread+conc": 0.354,
    "spread+edge": 0.341,
    "spread+hf": 0.358,
    "all_four": 0.352,
    "pattern_only": 0.109
  },
  "auc_error": {
    "spread": 0.9259259259259259,
    "concentration": 0.7081481481481482,
    "edge_ratio": 0.7777777777777778,
    "hf_fraction": 0.7377777777777778
  },
  "auc_miscal": {
    "spread": 0.5051851851851852,
    "concentration": 0.6637037037037037,
    "edge_ratio": 0.6385185185185185,
    "hf_fraction": 0.6918518518518518
  },
  "thresholds": {
    "error": 16.2128537495931,
    "ratio": 0.668415688196936
  },
  "within_level_spread_corr": {
    "FFHQ faces": 0.945742014394545,
    "CelebA portraits": 0.8959271670332966,
    "Animal faces": 0.8540147382537663,
    "Painted portraits": 0.8176000081539417,
    "Single objects": 0.4